# 40 — Rilevamento one-class a rischio fissato

Applica al CSV la parte del protocollo del relatore che il CSV permette.

- **Training sui soli benigni** (Mahalanobis con shrinkage, residuo PCA, Isolation Forest, One-Class SVM, Mahalanobis su random Fourier features). Un Random Forest supervisionato resta solo come riferimento.
- **Soglia conforme** calcolata su righe benigne di calibrazione, ai livelli α = 1%, 0,1%, 0,01%. Sul test: FPR osservato, limite superiore al 95% e TPR.
- **Ablation (E3):** con e senza `noise` e `total_gain_db` (possibili indicatori del dominio di acquisizione), con e senza `freq1` (il canale, che la specifica usa per stratificare e non come input).
- **Feature di ordine superiore per riga:** prodotti di secondo e terzo ordine tra le feature della stessa riga.
- **Strati (E2):** solo per canale (`freq1`).

**Limiti del CSV, da dichiarare:**
- non ci sono file né sessioni: l'unità di rischio è la riga e l'indipendenza tra righe è un'ipotesi non verificata;
- le righe sono rimescolate: niente finestre, momenti temporali, budget di report (E4) o rilevamento sequenziale (E5);
- mancano modalità di scansione, potenza, forma d'onda e fondo: niente strati su queste variabili.

In [1]:
import os, sys, time, json
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np, pandas as pd
import qdetect as qd
from qdetect import oneclass as O
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)

DATA_DIR = Path(os.environ.get('QDETECT_DATA_DIR', 'output/split')).expanduser()
TRAIN_CSV, TEST_CSV = DATA_DIR / 'train_dataset.csv', DATA_DIR / 'test_dataset.csv'
SPLIT = REPO / 'splits' / 'split_oneclass_v1.npz'
RESULTS = REPO / 'results' / 'oneclass_risk.csv'

## 1. Split più grande per il rischio basso

Per stimare soglie a α = 0,01% servono almeno 9.999 righe benigne di calibrazione. Per *certificare* FPR ≤ α con zero falsi allarmi servono 299, 2.995 e 29.956 negativi indipendenti sul test. Lo split usa quindi 50.000 righe di training, 20.000 di calibrazione e 60.000 di test nuovo (30.000 benigne).

In [2]:
print({a: O.min_negatives_for_zero_fp(a) for a in O.ALPHAS})
if SPLIT.exists():
    split = qd.load_split(SPLIT)
else:
    t0 = time.time()
    hashes = qd.vector_hashes(TRAIN_CSV)
    tr, s1 = qd.sample_per_class(TRAIN_CSV, 120_000, seed=2026)
    te, s2 = qd.sample_per_class(TEST_CSV, 60_000, seed=2027, exclude_groups=hashes)
    orig, s3 = qd.sample_per_class(TEST_CSV, 30_000, seed=2028)
    split = qd.make_split_from_files(tr, te, orig, n_train=5_000, n_train_large=50_000,
                                     n_val=20_000, n_test=60_000, seed=42)
    qd.save_split(split, SPLIT, extra_meta={'sampling': {'train': s1, 'test': s2, 'test_original': s3}})
    print(f'split creato in {time.time()-t0:.0f} s')
display(split.summary())

{0.01: 299, 0.001: 2995, 0.0001: 29956}


,set,n,frac_malicious
0,train,5000,0.49940
1,train_large,50000,0.49946
2,val,20000,0.50485
3,test,60000,0.50000
4,test_original,60000,0.50000


## 2. Griglia: insiemi di feature × rappresentazione × modello

In [3]:
names = split.features
tr, cal = split.idx_train_large, split.idx_val
f_idx = names.index('freq1')
tests = {'test nuovo': (split.X[split.idx_test], split.y[split.idx_test], split.X[split.idx_test, f_idx]),
         'test originale': (split.X[split.extra['test_original']], split.y[split.extra['test_original']],
                            split.X[split.extra['test_original'], f_idx])}
t0 = time.time()
res = O.run_grid(split.X[tr], split.y[tr], split.X[cal], split.y[cal], tests, names)
RESULTS.parent.mkdir(exist_ok=True); res.to_csv(RESULTS, index=False)
print(f'totale {time.time()-t0:.0f} s, {len(res)} righe')

all8   first  mahalanobis       test nuovo: AUROC=0.740 TPR@1%=0.636  test originale: AUROC=0.972 TPR@1%=0.751  (0.5s)


all8   first  pca_residual      test nuovo: AUROC=0.522 TPR@1%=0.142  test originale: AUROC=0.638 TPR@1%=0.249  (0.5s)


all8   first  isolation_forest  test nuovo: AUROC=0.678 TPR@1%=0.398  test originale: AUROC=0.935 TPR@1%=0.471  (1.2s)


all8   first  ocsvm_rbf         test nuovo: AUROC=0.741 TPR@1%=0.570  test originale: AUROC=0.958 TPR@1%=0.710  (0.9s)


all8   first  rff_mahalanobis   test nuovo: AUROC=0.764 TPR@1%=0.648  test originale: AUROC=0.971 TPR@1%=0.714  (2.3s)


all8   first  supervised_rf     test nuovo: AUROC=0.984 TPR@1%=0.916  test originale: AUROC=0.999 TPR@1%=0.984  (0.8s)


... (45 outputs left out)

core5  third  ocsvm_rbf         test nuovo: AUROC=0.737 TPR@1%=0.561  test originale: AUROC=0.945 TPR@1%=0.715  (0.9s)


core5  third  rff_mahalanobis   test nuovo: AUROC=0.747 TPR@1%=0.597  test originale: AUROC=0.962 TPR@1%=0.715  (2.2s)


core5  third  supervised_rf     test nuovo: AUROC=0.919 TPR@1%=0.707  test originale: AUROC=0.996 TPR@1%=0.931  (0.8s)


totale 93 s, 2592 righe


## 3. Risultati a rischio fissato (test nuovo, tutte le righe)

`fpr_certified` = il limite superiore al 95% del falso allarme è ≤ α (sempre assumendo righe indipendenti).

In [4]:
A = res[(res.stratum == 'all')]
show = ['feature_set', 'representation', 'model', 'training', 'auroc', 'tpr', 'fpr', 'fpr_upper95', 'fpr_certified', 'finite_threshold']
for alpha in O.ALPHAS:
    t = A[(A.test == 'test nuovo') & (A.alpha == alpha)].sort_values('tpr', ascending=False)
    print(f'\n=== α = {alpha:g} ===')
    display(t[show].head(12).round(4))


=== α = 0.01 ===


,feature_set,representation,model,training,auroc,tpr,fpr,fpr_upper95,fpr_certified,finite_threshold
240,all8,first,supervised_rf,supervised,0.9838,0.9160,0.0510,0.0531,False,True
528,all8,second,supervised_rf,supervised,0.9848,0.9143,0.0442,0.0462,False,True
816,all8,third,supervised_rf,supervised,0.9835,0.9078,0.0440,0.0460,False,True
1104,all7,first,supervised_rf,supervised,0.9549,0.8222,0.0722,0.0747,False,True
1392,all7,second,supervised_rf,supervised,0.9530,0.8193,0.0678,0.0702,False,True
1680,all7,third,supervised_rf,supervised,0.9508,0.8189,0.0703,0.0728,False,True
1968,core5,first,supervised_rf,supervised,0.9233,0.7319,0.0814,0.0841,False,True
2256,core5,second,supervised_rf,supervised,0.9202,0.7158,0.0748,0.0773,False,True
2544,core5,third,supervised_rf,supervised,0.9187,0.7069,0.0747,0.0772,False,True
480,all8,second,rff_mahalanobis,benign_only,0.7363,0.6521,0.2764,0.2807,False,True



=== α = 0.001 ===


,feature_set,representation,model,training,auroc,tpr,fpr,fpr_upper95,fpr_certified,finite_threshold
248,all8,first,supervised_rf,supervised,0.9838,0.3761,0.0010,0.0014,False,True
536,all8,second,supervised_rf,supervised,0.9848,0.3313,0.0007,0.0010,False,True
824,all8,third,supervised_rf,supervised,0.9835,0.2752,0.0005,0.0008,True,True
1928,core5,first,rff_mahalanobis,benign_only,0.7733,0.2604,0.0330,0.0347,False,True
200,all8,first,rff_mahalanobis,benign_only,0.7637,0.2592,0.0414,0.0434,False,True
1112,all7,first,supervised_rf,supervised,0.9549,0.2381,0.0066,0.0074,False,True
1400,all7,second,supervised_rf,supervised,0.9530,0.2094,0.0061,0.0069,False,True
1064,all7,first,rff_mahalanobis,benign_only,0.7588,0.1987,0.0222,0.0237,False,True
1448,all7,third,mahalanobis,benign_only,0.7555,0.1959,0.0279,0.0295,False,True
2312,core5,third,mahalanobis,benign_only,0.7496,0.1902,0.0244,0.0260,False,True



=== α = 0.0001 ===


,feature_set,representation,model,training,auroc,tpr,fpr,fpr_upper95,fpr_certified,finite_threshold
16,all8,first,mahalanobis,benign_only,0.7399,0.0,0.0,0.0001,True,False
1936,core5,first,rff_mahalanobis,benign_only,0.7733,0.0,0.0,0.0001,True,False
1408,all7,second,supervised_rf,supervised,0.9530,0.0,0.0,0.0001,True,False
1456,all7,third,mahalanobis,benign_only,0.7555,0.0,0.0,0.0001,True,False
1504,all7,third,pca_residual,benign_only,0.7509,0.0,0.0,0.0001,True,False
1552,all7,third,isolation_forest,benign_only,0.7337,0.0,0.0,0.0001,True,False
1600,all7,third,ocsvm_rbf,benign_only,0.7430,0.0,0.0,0.0001,True,False
1648,all7,third,rff_mahalanobis,benign_only,0.7482,0.0,0.0,0.0001,True,False
1696,all7,third,supervised_rf,supervised,0.9508,0.0,0.0,0.0001,True,False
1744,core5,first,mahalanobis,benign_only,0.7775,0.0,0.0,0.0001,True,False


## 4. Ablation: canale, rumore e guadagno; ordine superiore

In [5]:
B = A[(A.test == 'test nuovo') & (A.training == 'benign_only')]
abl = B.pivot_table(index=['model', 'representation', 'alpha'], columns='feature_set', values='tpr').reset_index()
abl['effetto freq1 (all8−all7)'] = abl['all8'] - abl['all7']
abl['effetto noise+gain (all7−core5)'] = abl['all7'] - abl['core5']
display(abl[abl.alpha == 0.01].round(3))
order = B.pivot_table(index=['model', 'feature_set', 'alpha'], columns='representation', values='tpr').reset_index()
order['second−first'] = order['second'] - order['first']; order['third−second'] = order['third'] - order['second']
display(order[order.alpha == 0.01].round(3))
auc = B[B.alpha == 0.01].pivot_table(index='model', columns=['feature_set', 'representation'], values='auroc')
display(auc.round(3))

feature_set,model,representation,alpha,all7,all8,core5,effetto freq1 (all8−all7),effetto noise+gain (all7−core5)
2,isolation_forest,first,0.01,0.521,0.398,0.547,-0.123,-0.025
5,isolation_forest,second,0.01,0.415,0.411,0.503,-0.004,-0.088
8,isolation_forest,third,0.01,0.375,0.388,0.437,0.013,-0.062
11,mahalanobis,first,0.01,0.628,0.636,0.634,0.008,-0.006
14,mahalanobis,second,0.01,0.642,0.622,0.631,-0.019,0.011
17,mahalanobis,third,0.01,0.583,0.610,0.573,0.027,0.009
20,ocsvm_rbf,first,0.01,0.547,0.570,0.550,0.023,-0.004
23,ocsvm_rbf,second,0.01,0.590,0.627,0.595,0.037,-0.005
26,ocsvm_rbf,third,0.01,0.565,0.611,0.561,0.046,0.004
29,pca_residual,first,0.01,0.151,0.142,0.459,-0.010,-0.307


representation,model,feature_set,alpha,first,second,third,second−first,third−second
2,isolation_forest,all7,0.01,0.521,0.415,0.375,-0.107,-0.040
5,isolation_forest,all8,0.01,0.398,0.411,0.388,0.013,-0.023
8,isolation_forest,core5,0.01,0.547,0.503,0.437,-0.044,-0.066
11,mahalanobis,all7,0.01,0.628,0.642,0.583,0.014,-0.059
14,mahalanobis,all8,0.01,0.636,0.622,0.610,-0.014,-0.012
17,mahalanobis,core5,0.01,0.634,0.631,0.573,-0.003,-0.057
20,ocsvm_rbf,all7,0.01,0.547,0.590,0.565,0.043,-0.025
23,ocsvm_rbf,all8,0.01,0.570,0.627,0.611,0.057,-0.016
26,ocsvm_rbf,core5,0.01,0.550,0.595,0.561,0.045,-0.034
29,pca_residual,all7,0.01,0.151,0.518,0.524,0.367,0.006


feature_set        all7                 all8                core5              
representation    first second  third  first second  third  first second  third
model                                                                          
isolation_forest  0.744  0.739  0.734  0.678  0.692  0.695  0.752  0.735  0.719
mahalanobis       0.771  0.766  0.756  0.740  0.743  0.739  0.777  0.769  0.750
ocsvm_rbf         0.738  0.743  0.743  0.741  0.735  0.730  0.743  0.744  0.737
pca_residual      0.526  0.733  0.751  0.522  0.733  0.733  0.668  0.737  0.734
rff_mahalanobis   0.759  0.747  0.748  0.764  0.736  0.730  0.773  0.750  0.747

## 5. Test nuovo contro test originale (stesso modello, stessa soglia)

In [6]:
C = A[(A.alpha == 0.01) & (A.training == 'benign_only')]
cmp_ = C.pivot_table(index=['feature_set', 'representation', 'model'], columns='test', values=['auroc', 'tpr']).round(3)
display(cmp_)

auroc                       tpr               
test                                        test nuovo test originale test nuovo test originale
feature_set representation model                                                               
all7        first          isolation_forest      0.744          0.952      0.521          0.693
                           mahalanobis           0.771          0.980      0.628          0.744
                           ocsvm_rbf             0.738          0.957      0.547          0.704
                           pca_residual          0.526          0.637      0.151          0.241
                           rff_mahalanobis       0.759          0.967      0.636          0.709
            second         isolation_forest      0.739          0.938      0.415          0.566
                           mahalanobis           0.766          0.972      0.642          0.731
                           ocsvm_rbf             0.743          0.946      0.590          0.716
                           pca_residual          0.733          0.958      0.518          0.679
                           rff_mahalanobis       0.747          0.959      0.632          0.719
            third          isolation_forest      0.734          0.896      0.375          0.522
                           mahalanobis           0.756          0.961      0.583          0.706
                           ocsvm_rbf             0.743          0.928      0.565          0.712
                           pca_residual          0.751          0.949      0.524          0.698
                           rff_mahalanobis       0.748          0.954      0.607          0.717
all8        first          isolation_forest      0.678          0.935      0.398          0.471
                           mahalanobis           0.740          0.972      0.636          0.751
                           ocsvm_rbf             0.741          0.958      0.570          0.710
                           pca_residual          0.522          0.638      0.142          0.249
                           rff_mahalanobis       0.764          0.971      0.648          0.714
            second         isolation_forest      0.692          0.929      0.411          0.488
                           mahalanobis           0.743          0.970      0.622          0.732
                           ocsvm_rbf             0.735          0.950      0.627          0.715
                           pca_residual          0.733          0.959      0.562          0.710
                           rff_mahalanobis       0.736          0.959      0.652          0.727
            third          isolation_forest      0.695          0.902      0.388          0.470
                           mahalanobis           0.739          0.960      0.610          0.707
                           ocsvm_rbf             0.730          0.938      0.611          0.711
                           pca_residual          0.733          0.940      0.616          0.726
                           rff_mahalanobis       0.730          0.955      0.621          0.708
core5       first          isolation_forest      0.752          0.952      0.547          0.702
                           mahalanobis           0.777          0.982      0.634          0.751
                           ocsvm_rbf             0.743          0.963      0.550          0.706
                           pca_residual          0.668          0.887      0.459          0.429
                           rff_mahalanobis       0.773          0.973      0.640          0.712
            second         isolation_forest      0.735          0.942      0.503          0.693
                           mahalanobis           0.769          0.971      0.631          0.727
                           ocsvm_rbf             0.744          0.950      0.595          0.722
                           pca_residual          0.737          0.947      0.562          0.684
                 

## 6. Strati per canale (miglior modello benign-only a α = 1% sul test nuovo)

In [7]:
best = B[B.alpha == 0.01].sort_values('tpr', ascending=False).iloc[0]
print('migliore:', best[['feature_set', 'representation', 'model', 'tpr', 'fpr']].to_dict())
S = res[(res.test == 'test nuovo') & (res.feature_set == best.feature_set) & (res.representation == best.representation)
        & (res.model == best.model) & (res.alpha == 0.01)]
display(S[['stratum', 'n_pos', 'tpr', 'auroc']].round(4))

migliore: {'feature_set': 'all8', 'representation': 'second', 'model': 'rff_mahalanobis', 'tpr': 0.6521333333333333, 'fpr': 0.2764333333333333}


,stratum,n_pos,tpr,auroc
480,all,30000,0.6521,0.7363
481,freq1=5180,3034,0.8530,0.8253
482,freq1=5200,2881,0.7626,0.7942
483,freq1=5240,2598,0.6925,0.7764
484,freq1=5745,1502,0.0646,0.3859
485,freq1=5765,2734,0.1913,0.5293
486,freq1=5785,3715,0.5300,0.6567
487,freq1=5805,13536,0.7677,0.7987


## 7. Gate di decisione (come nella specifica)

Per ogni combinazione si considerano solo le soglie con FPR certificato ≤ α. Se il miglior TPR è ≥ 0,95 → **CLASSICAL_SATURATION_CANDIDATE**: in quel regime non si cerca un vantaggio predittivo. Se è più basso → **DIFFICULTY_CANDIDATE_NOT_CERTIFIED**. Se nessuna soglia è certificata → **INSUFFICIENT_RISK_EVIDENCE**.

In [8]:
rows = []
for (fs, alpha), g in A[(A.test == 'test nuovo') & (A.training == 'benign_only')].groupby(['feature_set', 'alpha']):
    ok = g[g.fpr_certified & g.finite_threshold & (g.tpr > 0)]
    status = ('INSUFFICIENT_RISK_EVIDENCE' if ok.empty else
              'CLASSICAL_SATURATION_CANDIDATE' if ok.tpr.max() >= 0.95 else 'DIFFICULTY_CANDIDATE_NOT_CERTIFIED')
    top = ok.sort_values('tpr').iloc[-1] if len(ok) else None
    rows.append(dict(feature_set=fs, alpha=alpha, status=status,
                     best_tpr=None if top is None else round(top.tpr, 4),
                     best=None if top is None else f'{top.model} / {top.representation}'))
gate = pd.DataFrame(rows); gate.to_csv(REPO / 'results' / 'oneclass_gate.csv', index=False); gate

,feature_set,alpha,status,best_tpr,best
0,all7,0.0001,INSUFFICIENT_RISK_EVIDENCE,None,None
1,all7,0.0010,INSUFFICIENT_RISK_EVIDENCE,None,None
2,all7,0.0100,INSUFFICIENT_RISK_EVIDENCE,None,None
3,all8,0.0001,INSUFFICIENT_RISK_EVIDENCE,None,None
4,all8,0.0010,INSUFFICIENT_RISK_EVIDENCE,None,None
5,all8,0.0100,INSUFFICIENT_RISK_EVIDENCE,None,None
6,core5,0.0001,INSUFFICIENT_RISK_EVIDENCE,None,None
7,core5,0.0010,INSUFFICIENT_RISK_EVIDENCE,None,None
8,core5,0.0100,INSUFFICIENT_RISK_EVIDENCE,None,None


## 8. Calibrazione nello stesso dominio del test nuovo

La sezione 3 mostra che sul test nuovo il falso allarme reale è circa 20 volte α. Le righe benigne di calibrazione vengono dal file di training (vettori frequenti), quelle del test nuovo sono vettori rari. Lo scambio tra calibrazione e test, ipotesi della soglia conforme, quindi non regge: per un modello one-class un vettore benigno *raro* è già un'anomalia.

Per separare i due effetti si ricalibra su righe **benigne** del test nuovo **non usate nel test**, con vettori diversi da quelli del test. Non si usa nessuna riga malevola e nessuna etichetta del test. È una scelta di adattamento al dominio e va dichiarata come tale: mostra quanto del problema è calibrazione e quanto è capacità di separare.

In [9]:
g = qd.feature_groups(split.X)
src = split.source
used = np.zeros(len(split.y), bool); used[split.idx_test] = True; used[split.extra['test_original']] = True
test_groups = np.unique(g[split.idx_test])
cand = np.flatnonzero((src == 1) & ~used & (split.y == 0) & ~np.isin(g, test_groups))
cal_new = np.random.default_rng(7).choice(cand, min(20_000, len(cand)), replace=False)
print('righe benigne di calibrazione nel dominio nuovo:', len(cal_new), 'su', len(cand), 'disponibili')
res2 = O.run_grid(split.X[tr], split.y[tr], split.X[cal_new], split.y[cal_new], {'test nuovo': tests['test nuovo']}, names, verbose=False)
res2.to_csv(REPO / 'results' / 'oneclass_risk_cal_nuovo.csv', index=False)
A2 = res2[res2.stratum == 'all']
for alpha in O.ALPHAS:
    print(f'\n=== α = {alpha:g}, calibrazione nel dominio nuovo ===')
    display(A2[A2.alpha == alpha].sort_values('tpr', ascending=False)[show].head(10).round(4))

righe benigne di calibrazione nel dominio nuovo: 20000 su 29415 disponibili



=== α = 0.01, calibrazione nel dominio nuovo ===


,feature_set,representation,model,training,auroc,tpr,fpr,fpr_upper95,fpr_certified,finite_threshold
264,all8,second,supervised_rf,supervised,0.9848,0.7193,0.0104,0.0115,False,True
120,all8,first,supervised_rf,supervised,0.9838,0.7021,0.0101,0.0111,False,True
408,all8,third,supervised_rf,supervised,0.9835,0.6750,0.0090,0.0099,True,True
552,all7,first,supervised_rf,supervised,0.9549,0.3103,0.0090,0.0099,True,True
696,all7,second,supervised_rf,supervised,0.9530,0.2844,0.0094,0.0103,False,True
840,all7,third,supervised_rf,supervised,0.9508,0.2687,0.0101,0.0111,False,True
984,core5,first,supervised_rf,supervised,0.9233,0.2200,0.0105,0.0116,False,True
1128,core5,second,supervised_rf,supervised,0.9202,0.1912,0.0105,0.0116,False,True
1272,core5,third,supervised_rf,supervised,0.9187,0.1775,0.0099,0.0109,False,True
960,core5,first,rff_mahalanobis,benign_only,0.7733,0.1672,0.0115,0.0125,False,True



=== α = 0.001, calibrazione nel dominio nuovo ===


,feature_set,representation,model,training,auroc,tpr,fpr,fpr_upper95,fpr_certified,finite_threshold
128,all8,first,supervised_rf,supervised,0.9838,0.3983,0.0014,0.0018,False,True
272,all8,second,supervised_rf,supervised,0.9848,0.3666,0.0009,0.0012,False,True
416,all8,third,supervised_rf,supervised,0.9835,0.3010,0.0008,0.0011,False,True
560,all7,first,supervised_rf,supervised,0.9549,0.0974,0.0012,0.0016,False,True
104,all8,first,rff_mahalanobis,benign_only,0.7637,0.0533,0.0011,0.0015,False,True
320,all8,third,pca_residual,benign_only,0.7333,0.0512,0.0007,0.0010,True,True
512,all7,first,ocsvm_rbf,benign_only,0.7382,0.0484,0.0008,0.0012,False,True
752,all7,third,pca_residual,benign_only,0.7509,0.0478,0.0007,0.0010,False,True
296,all8,third,mahalanobis,benign_only,0.7390,0.0476,0.0008,0.0011,False,True
1160,core5,third,mahalanobis,benign_only,0.7496,0.0476,0.0010,0.0013,False,True



=== α = 0.0001, calibrazione nel dominio nuovo ===


,feature_set,representation,model,training,auroc,tpr,fpr,fpr_upper95,fpr_certified,finite_threshold
136,all8,first,supervised_rf,supervised,0.9838,0.1525,0.0000,0.0001,True,True
112,all8,first,rff_mahalanobis,benign_only,0.7637,0.0128,0.0001,0.0003,False,True
544,all7,first,rff_mahalanobis,benign_only,0.7588,0.0115,0.0001,0.0002,False,True
976,core5,first,rff_mahalanobis,benign_only,0.7733,0.0065,0.0002,0.0004,False,True
328,all8,third,pca_residual,benign_only,0.7333,0.0055,0.0001,0.0003,False,True
760,all7,third,pca_residual,benign_only,0.7509,0.0050,0.0001,0.0003,False,True
520,all7,first,ocsvm_rbf,benign_only,0.7382,0.0019,0.0000,0.0001,True,True
1216,core5,third,isolation_forest,benign_only,0.7186,0.0010,0.0000,0.0002,False,True
952,core5,first,ocsvm_rbf,benign_only,0.7432,0.0009,0.0000,0.0001,True,True
88,all8,first,ocsvm_rbf,benign_only,0.7407,0.0007,0.0000,0.0001,True,True


In [10]:
cmp2 = (A[(A.test == 'test nuovo') & (A.representation == 'first')].set_index(['feature_set', 'model', 'alpha'])[['fpr', 'tpr']]
        .join(A2[A2.representation == 'first'].set_index(['feature_set', 'model', 'alpha'])[['fpr', 'tpr', 'fpr_certified']], rsuffix=' (cal. nuova)'))
display(cmp2.reset_index().query('alpha == 0.01').round(4))
rows = []
for (fs, alpha), gg in A2[A2.training == 'benign_only'].groupby(['feature_set', 'alpha']):
    ok = gg[gg.fpr_certified & gg.finite_threshold & (gg.tpr > 0)]
    status = ('INSUFFICIENT_RISK_EVIDENCE' if ok.empty else
              'CLASSICAL_SATURATION_CANDIDATE' if ok.tpr.max() >= 0.95 else 'DIFFICULTY_CANDIDATE_NOT_CERTIFIED')
    top = ok.sort_values('tpr').iloc[-1] if len(ok) else None
    rows.append(dict(feature_set=fs, alpha=alpha, status=status, best_tpr=None if top is None else round(top.tpr, 4),
                     best=None if top is None else f'{top.model} / {top.representation}'))
gate2 = pd.DataFrame(rows); gate2.to_csv(REPO / 'results' / 'oneclass_gate_cal_nuovo.csv', index=False); gate2

/var/folders/16/pbm4xp7d0_n5h55kymnw83mc0000gn/T/ipykernel_34572/1786480038.py:2: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  .join(A2[A2.representation == 'first'].set_index(['feature_set', 'model', 'alpha'])[['fpr', 'tpr', 'fpr_certified']], rsuffix=' (cal. nuova)'))


,feature_set,model,alpha,fpr,tpr,fpr (cal. nuova),tpr (cal. nuova),fpr_certified
0,all8,mahalanobis,0.01,0.2528,0.6360,0.0104,0.0632,False
3,all8,pca_residual,0.01,0.0527,0.1416,0.0090,0.0360,True
6,all8,isolation_forest,0.01,0.1908,0.3980,0.0089,0.0028,True
9,all8,ocsvm_rbf,0.01,0.2408,0.5699,0.0102,0.1233,False
12,all8,rff_mahalanobis,0.01,0.2638,0.6485,0.0100,0.1571,False
15,all8,supervised_rf,0.01,0.0510,0.9160,0.0101,0.7021,False
18,all7,mahalanobis,0.01,0.2378,0.6278,0.0104,0.0687,False
21,all7,pca_residual,0.01,0.0519,0.1512,0.0095,0.0452,False
24,all7,isolation_forest,0.01,0.1922,0.5214,0.0093,0.0939,False
27,all7,ocsvm_rbf,0.01,0.2396,0.5467,0.0102,0.1265,False


,feature_set,alpha,status,best_tpr,best
0,all7,0.0001,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.0019,ocsvm_rbf / first
1,all7,0.0010,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.0255,rff_mahalanobis / first
2,all7,0.0100,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.1290,rff_mahalanobis / first
3,all8,0.0001,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.0007,ocsvm_rbf / first
4,all8,0.0010,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.0512,pca_residual / third
5,all8,0.0100,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.0360,pca_residual / first
6,core5,0.0001,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.0009,ocsvm_rbf / first
7,core5,0.0010,DIFFICULTY_CANDIDATE_NOT_CERTIFIED,0.0057,isolation_forest / third
8,core5,0.0100,INSUFFICIENT_RISK_EVIDENCE,NaN,NaN
